# 5.8 — Structure learning

Learning a graph means paying for edges only when their likelihood gain is worth the complexity.

Structure learning uses likelihood from Bayesian updating and graphical factorizations. It prepares model selection in HMMs, DBNs, and probabilistic programs where structure may be learned or designed.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(58)


def bic_edge_penalty(n, extra_params=1):
    return 0.5 * extra_params * math.log(n)


def score_graph_bic(data, parents):
    data = np.asarray(data, dtype=int)
    n, p = data.shape
    loglik = 0.0
    params = 0
    for child in range(p):
        par = tuple(parents.get(child, ()))
        if not par:
            count1 = data[:, child].sum()
            count0 = n - count1
            prob1 = (count1 + 1) / (n + 2)
            prob0 = 1 - prob1
            loglik += count1 * math.log(prob1) + count0 * math.log(prob0)
            params += 1
            continue
        parent_configs = list(itertools.product([0, 1], repeat=len(par)))
        params += len(parent_configs)
        for config in parent_configs:
            mask = np.ones(n, dtype=bool)
            for var, value in zip(par, config):
                mask = mask & (data[:, var] == value)
            subset = data[mask, child]
            count1 = subset.sum()
            count0 = subset.size - count1
            prob1 = (count1 + 1) / (subset.size + 2)
            prob0 = 1 - prob1
            loglik += count1 * math.log(prob1) + count0 * math.log(prob0)
    bic = loglik - 0.5 * params * math.log(n)
    return bic, loglik, params


def has_cycle(parents, p):
    visiting = set()
    visited = set()

    def dfs(node):
        if node in visiting:
            return True
        if node in visited:
            return False
        visiting.add(node)
        for parent in parents.get(node, ()): 
            if dfs(parent):
                return True
        visiting.remove(node)
        visited.add(node)
        return False

    return any(dfs(node) for node in range(p))


def greedy_structure_search(data, max_parents=2, restarts=1):
    p = data.shape[1]
    best_parents = {i: tuple() for i in range(p)}
    best_score, best_loglik, best_params = score_graph_bic(data, best_parents)
    history = [best_score]
    for restart in range(restarts):
        improved = True
        while improved:
            improved = False
            candidates = []
            for src in range(p):
                for dst in range(p):
                    if src == dst:
                        continue
                    current = tuple(best_parents.get(dst, ()))
                    if src in current or len(current) >= max_parents:
                        continue
                    trial = dict(best_parents)
                    trial[dst] = tuple(sorted(current + (src,)))
                    if has_cycle(trial, p):
                        continue
                    score, loglik, params = score_graph_bic(data, trial)
                    candidates.append((score, trial))
            if candidates:
                score, trial = max(candidates, key=lambda item: item[0])
                if score > best_score + 1e-9:
                    best_score = score
                    best_parents = trial
                    improved = True
            history.append(best_score)
    return best_parents, history


def edges_from_parents(parents):
    return {(parent, child) for child, plist in parents.items() for parent in plist}


def structural_hamming_distance(learned, truth):
    return len(edges_from_parents(learned) ^ edges_from_parents(truth))


def simulate_binary_dag(n, parents, weights):
    p = len(parents)
    data = np.zeros((n, p), dtype=int)
    order = list(range(p))
    for child in order:
        plist = parents.get(child, ())
        logits = np.full(n, weights.get((child, "bias"), 0.0))
        for parent in plist:
            logits += weights.get((parent, child), 1.0) * data[:, parent]
        probs = 1 / (1 + np.exp(-logits))
        data[:, child] = rng.binomial(1, probs)
    return data


def make_structure_ladder():
    d1_truth = {0: tuple(), 1: (0,)}
    d2_truth = {0: tuple(), 1: (0,), 2: (1,)}
    d3_truth = {0: tuple(), 1: (0,), 2: (0,)}
    d4_truth = {0: tuple(), 1: (0,), 2: (0, 1)}
    d5_truth = {0: tuple(), 1: (0,), 2: (0,), 3: (1, 2), 4: (2,), 5: tuple()}
    return [
        {"name": "D1 2-node edge", "data": simulate_binary_dag(100, d1_truth, {(0, 1): 2.2, (0, "bias"): -0.2, (1, "bias"): -0.8}), "truth": d1_truth},
        {"name": "D2 clean 3-node DAG", "data": simulate_binary_dag(160, d2_truth, {(0, 1): 2.0, (1, 2): 1.8, (0, "bias"): -0.1, (1, "bias"): -0.7, (2, "bias"): -0.6}), "truth": d2_truth},
        {"name": "D3 Markov-equivalent fork", "data": simulate_binary_dag(180, d3_truth, {(0, 1): 1.5, (0, 2): 1.5, (0, "bias"): 0.0, (1, "bias"): -0.6, (2, "bias"): -0.6}), "truth": d3_truth},
        {"name": "D4 contingency-table sample", "data": simulate_binary_dag(220, d4_truth, {(0, 1): 1.4, (0, 2): 0.9, (1, 2): 1.1, (0, "bias"): -0.4, (1, "bias"): -0.2, (2, "bias"): -1.0}), "truth": d4_truth},
        {"name": "D5 weak-spurious search space", "data": simulate_binary_dag(260, d5_truth, {(0, 1): 1.2, (0, 2): 1.0, (1, 3): 0.9, (2, 3): 0.9, (2, 4): 0.8, (0, "bias"): -0.3, (5, "bias"): 0.1}), "truth": d5_truth},
    ]


## The concept, built once (D1)

For a graph $G$ with fitted parameters $\hat\theta_G$, free parameter count $k_G$, and $n$ samples, the lesson uses

$$\mathrm{BIC}(G)=\log p(D\mid \hat\theta_G,G)-\frac{k_G}{2}\log n.$$

The worked gain is $0.177741$ nats per sample.

In [ ]:

n = 100
gain_per_sample = 0.177741
gain_total = n * gain_per_sample
penalty = bic_edge_penalty(n, extra_params=1)
print("gain per sample", gain_per_sample)
print("gain total", gain_total)
print("penalty", penalty)
assert np.isclose(gain_per_sample, 0.177741)
assert np.isclose(gain_total, 17.7741)
assert np.isclose(round(gain_total, 3), 17.774)
assert np.isclose(penalty, 2.302585092994046)
assert np.isclose(round(penalty, 3), 2.303)


A reusable scorer fits discrete conditional probability tables and subtracts the same BIC penalty. Strong edges survive; weak edges should lose.

In [ ]:

lesson_data = np.zeros((100, 2), dtype=int)
lesson_data[:50, 0] = 0
lesson_data[50:, 0] = 1
lesson_data[:10, 1] = 1
lesson_data[50:90, 1] = 1
empty = {0: tuple(), 1: tuple()}
edge = {0: tuple(), 1: (0,)}
empty_score, empty_ll, empty_params = score_graph_bic(lesson_data, empty)
edge_score, edge_ll, edge_params = score_graph_bic(lesson_data, edge)
print("example loglik gain", edge_ll - empty_ll)
print("example BIC gain", edge_score - empty_score)
assert edge_params - empty_params == 1
assert edge_score > empty_score


## The dataset ladder

D1-D5 are synthetic binary graphical models with known truth except that D4 mimics a small contingency-table sample. Complexity rises through more variables, equivalent candidates, and weak spurious edges.

In [ ]:

ladder = make_structure_ladder()
for rung in ladder:
    print(rung["name"], "shape", rung["data"].shape, "truth", sorted(edges_from_parents(rung["truth"])))
    print("sample rows", rung["data"][:5].tolist())


In [ ]:

rows = []
for rung in ladder:
    learned, history = greedy_structure_search(rung["data"], max_parents=2, restarts=3)
    shd = structural_hamming_distance(learned, rung["truth"])
    rows.append({"name": rung["name"], "learned": learned, "truth": rung["truth"], "history": history, "metric": shd})
for row in rows:
    print(row["name"], "SHD", row["metric"], "learned", sorted(edges_from_parents(row["learned"])))


In [ ]:

fig, axes = plt.subplots(2, len(rows), figsize=(16, 6))
for j, row in enumerate(rows):
    p = max(max(edge) for edge in edges_from_parents(row["truth"]) | edges_from_parents(row["learned"])) + 1
    truth_mat = np.zeros((p, p))
    learned_mat = np.zeros((p, p))
    for src, dst in edges_from_parents(row["truth"]):
        truth_mat[src, dst] = 1
    for src, dst in edges_from_parents(row["learned"]):
        learned_mat[src, dst] = 1
    axes[0, j].imshow(truth_mat - learned_mat, vmin=-1, vmax=1, cmap="coolwarm")
    axes[0, j].set_title(row["name"].split()[0])
    axes[1, j].plot(row["history"], marker="o")
    axes[1, j].set_title(f"SHD {row['metric']}")
    axes[1, j].set_xlabel("search iteration")
fig.suptitle("Learned graph differences and score path")
plt.tight_layout()


## Pitfall on the hardest rung

Greedy search without safeguards can reward tiny likelihood gains and get trapped. The fix is to keep the BIC penalty and use restarts or stricter gain thresholds.

In [ ]:

d5 = ladder[-1]
no_penalty_parents = {i: tuple() for i in range(d5["data"].shape[1])}
base_score, base_ll, base_params = score_graph_bic(d5["data"], no_penalty_parents)
weak_gain = 0.5
penalty = bic_edge_penalty(d5["data"].shape[0], extra_params=1)
print("illustrative weak likelihood gain", weak_gain)
print("BIC penalty", penalty)
print("weak edge accepted without penalty", weak_gain > 0)
print("weak edge accepted with BIC", weak_gain > penalty)
assert weak_gain < penalty
learned_fixed, fixed_history = greedy_structure_search(d5["data"], max_parents=2, restarts=5)
print("fixed learned edges", sorted(edges_from_parents(learned_fixed)))


## Evaluate it + Practice

- Metric: structural Hamming distance to the known graph
- No-skill baseline: empty graph with independent variables
- Cheap sanity check: score of the true strong edge beats the empty graph by more than 2.303
- Ablation: remove the BIC penalty and D5 accumulates weak edges
- Failure signals: many Markov-equivalent ties, tiny gains, or dense learned graphs


Practice: Reverse a Markov-equivalent D3 edge and compare BIC.

Practice: Raise n in D1 and recompute the penalty.

Practice: Limit max_parents to one on D5 and measure SHD.